# 04 - Weak labels and the stratified split

The satisfaction label is derived from the star rating (4-5 = Satisfied, 1-2 = Dissatisfied, 3 excluded). This notebook checks the mapping, then splits the modelling set 70/15/15, stratified jointly on label and language, with the random seed fixed. Exact duplicates were removed in notebook 02, before splitting, so no text can sit on both sides of a partition. The three partitions are written with their `review_id` so that predictions produced on a GPU machine (notebook 07) can be paired back row by row.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# make src importable when the kernel starts inside notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid", context="notebook")
from src import config
from src.io import ensure_dirs, save_table, update_metrics, read_metrics
ensure_dirs()

In [2]:
from src.labelling import weak_label_from_stars, stratified_split, check_split_leakage

modelling = pd.read_parquet(config.MODELLING_FILE)
check = modelling["star_rating"].map(weak_label_from_stars)
assert (check == modelling["satisfaction_label"]).all(), "label mapping mismatch"
print(pd.crosstab(modelling.star_rating, modelling.satisfaction_label))

satisfaction_label  Dissatisfied  Satisfied
star_rating                                
1                          23476          0
2                           3714          0
4                              0       3352
5                              0      18388


In [3]:
splits = stratified_split(modelling)
for k, v in splits.items():
    print(f"{k:5s} n={len(v):6,}  ", (v.satisfaction_label.value_counts(normalize=True) * 100).round(1).to_dict(),
          (v.language.value_counts(normalize=True) * 100).round(1).to_dict())
leak = check_split_leakage(splits)
print("\nleakage check:", leak)
assert all(v == 0 for v in leak.values())

train n=34,250   {'Dissatisfied': 55.6, 'Satisfied': 44.4} {'English': 75.7, 'Arabic': 24.3}
val   n= 7,340   {'Dissatisfied': 55.6, 'Satisfied': 44.4} {'English': 75.7, 'Arabic': 24.3}
test  n= 7,340   {'Dissatisfied': 55.6, 'Satisfied': 44.4} {'English': 75.7, 'Arabic': 24.3}

leakage check: {'train-val_id_overlap': 0, 'train-val_text_overlap': 0, 'train-test_id_overlap': 0, 'train-test_text_overlap': 0, 'val-test_id_overlap': 0, 'val-test_text_overlap': 0}


In [4]:
summary = pd.DataFrame([
    {"split": k, "n": len(v),
     "Dissatisfied": int((v.satisfaction_label == "Dissatisfied").sum()),
     "Satisfied": int((v.satisfaction_label == "Satisfied").sum()),
     "Arabic": int((v.language == "Arabic").sum()),
     "English": int((v.language == "English").sum())}
    for k, v in splits.items()])
save_table(summary, "split_summary")
for k, v in splits.items():
    v.to_parquet(config.SPLIT_FILES[k], index=False)
update_metrics("split", {"summary": summary.to_dict(orient="records"), "leakage": leak,
                         "seed": config.RANDOM_SEED})
summary

,split,n,Dissatisfied,Satisfied,Arabic,English
0,train,34250,19032,15218,8308,25942
1,val,7340,4079,3261,1780,5560
2,test,7340,4079,3261,1781,5559
